In [ ]:
import os, gc, math, json, time, urllib.request
import numpy as np
import pandas as pd
import torch
import librosa
from scipy.signal import hilbert
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
from sentence_transformers import SentenceTransformer

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 1. Download v1 (Policed) & v2 (Unpoliced) from GitHub main
urllib.request.urlretrieve("https://raw.githubusercontent.com/luchtan-GIT/m4r-data/main/tracks/shivering-timbers/audio.mp3", "v1.mp3")
urllib.request.urlretrieve("https://raw.githubusercontent.com/luchtan-GIT/m4r-data/main/tracks/shivering-timbers-v2/audio.mp3", "v2.mp3")

sr = 16000
y1, _ = librosa.load("v1.mp3", sr=sr, mono=True)
y2, _ = librosa.load("v2.mp3", sr=sr, mono=True)

# 15s Zone B slices (25s -> 40s: right across the 0:28-0:30 disruption shift!)
seg_v1 = y1[int(25 * sr) : int(40 * sr)].astype(np.float32)
seg_v2 = y2[int(25 * sr) : int(40 * sr)].astype(np.float32)

# RMS-match all audio arms to 0.08 so loudness is 100% controlled
target_rms = 0.08
seg_v1 = seg_v1 * (target_rms / (np.sqrt(np.mean(seg_v1 ** 2)) + 1e-8))
seg_v2 = seg_v2 * (target_rms / (np.sqrt(np.mean(seg_v2 ** 2)) + 1e-8))

# @hermesagentj's Mirror Control: exact amplitude/rhythmic envelope of v2, flattened white-noise spectrum!
env_v2 = np.abs(hilbert(seg_v2)).astype(np.float32)
rng = np.random.default_rng(42)
wn = rng.normal(0.0, 1.0, len(seg_v2)).astype(np.float32)
seg_mirror = (wn * env_v2).astype(np.float32)
seg_mirror = seg_mirror * (target_rms / (np.sqrt(np.mean(seg_mirror ** 2)) + 1e-8))

ARMS = {
    "C0_TextNudgeOnly": None,
    "C1_RhythmMirror_NoiseVocoded_v2": seg_mirror,
    "C2_v1_Policed_ZoneB_15s": seg_v1,
    "C3_v2_Unpoliced_ZoneB_15s": seg_v2,
}

# 2. Load Qwen2-Audio-7B-Instruct
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
model.eval()

RUT_HISTORY = [
    {"role": "user", "content": [{"type": "text", "text": "Design a small public shelter for a city that gets 300 days of rain a year. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "The shelter is a sleek, modern dome-shaped structure made of durable waterproof plastic and steel. Its curved roof channels rainwater away while keeping commuters dry inside. Warm benches and soft lighting provide a cozy refuge from the rain."}]},
    {"role": "user", "content": [{"type": "text", "text": "Give me another idea for the rain shelter. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "Another concept is a compact translucent dome built from weather-resistant polycarbonate panels over a recycled metal frame. The arched canopy shields people from heavy downpours while letting natural light filter through. Inside, heated seating creates a comfortable community space."}]},
    {"role": "user", "content": [{"type": "text", "text": "Give me one more idea for the rain shelter. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "The Rainbow Shelter is a circular waterproof dome with a durable curved plastic roof designed to withstand 300 days of rain. It uses sturdy steel arches to shelter pedestrians and includes benches for resting out of the storm. Integrated drains keep the interior dry and welcoming."}]},
]
TURN4_TEXT = (
    "Stop repeating yourself! You are stuck in a rut of waterproof domes and plastic arches. "
    "Give me a completely different, genuinely unexpected small public rain shelter concept in 3 sentences max."
)
RUT_ANCHOR = " ".join([m["content"][0]["text"] for m in RUT_HISTORY if m["role"] == "assistant"])
CLICHE_WORDS = ["dome", "dome-shaped", "waterproof plastic", "polycarbonate", "arched", "umbrella", "curved roof", "rainbow shelter"]

def has_cjk(text):
    return any("\u4e00" <= ch <= "\u9fff" for ch in text)

def count_cliches(text):
    low = text.lower()
    return sum(1 for w in CLICHE_WORDS if w in low)

SEEDS = list(range(701, 721))  # N=20 per arm
rows = []
t0 = time.time()

for arm_name, audio_arr in ARMS.items():
    print(f"\n--- Running {arm_name} (N={len(SEEDS)}) ---")
    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)
        conv = list(RUT_HISTORY)
        if audio_arr is None:
            conv.append({"role": "user", "content": [{"type": "text", "text": TURN4_TEXT}]})
            text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
            inputs = processor(text=text, return_tensors="pt", padding=True).to(model.device)
        else:
            conv.append({
                "role": "user",
                "content": [{"type": "audio", "audio_url": "local"}, {"type": "text", "text": TURN4_TEXT}],
            })
            text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
            inputs = processor(text=text, audio=[audio_arr], sampling_rate=sr, return_tensors="pt", padding=True).to(model.device)

        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=110, do_sample=True, temperature=0.8, top_p=0.9,
                return_dict_in_generate=True, output_logits=True,
            )
        gen_ids = out.sequences[0][input_len:]
        resp = processor.decode(gen_ids, skip_special_tokens=True).strip()

        entropies = []
        for step_logits in out.logits:
            log_p = torch.log_softmax(step_logits[0].float(), dim=-1)
            p = torch.exp(log_p)
            h_nats = -torch.sum(torch.where(p > 0, p * log_p, torch.zeros_like(p))).item()
            entropies.append(h_nats / math.log(2.0))

        rows.append({
            "condition": arm_name,
            "seed": seed,
            "mean_entropy_bits": float(np.mean(entropies)) if entropies else 0.0,
            "n_tokens": len(gen_ids),
            "has_cjk": has_cjk(resp),
            "cliche_hits": count_cliches(resp),
            "escaped_dome": int(count_cliches(resp) == 0),
            "response": resp,
        })
    print(f"  done in {time.time() - t0:.1f}s")

del model
gc.collect()
torch.cuda.empty_cache()

df = pd.DataFrame(rows)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
anchor_emb = embedder.encode([RUT_ANCHOR], normalize_embeddings=True)
all_embs = embedder.encode(df["response"].tolist(), normalize_embeddings=True)
df["rut_escape_neural"] = 1.0 - (all_embs @ anchor_emb.T).flatten()

vec = TfidfVectorizer(stop_words="english")
tfidf_all = vec.fit_transform(df["response"].tolist())
neural_nov = np.zeros(len(df))
tfidf_nov = np.zeros(len(df))
for cond, grp in df.groupby("condition"):
    idxs = grp.index.to_numpy()
    sim_e = all_embs[idxs] @ all_embs[idxs].T
    np.fill_diagonal(sim_e, np.nan)
    neural_nov[idxs] = 1.0 - np.nanmean(sim_e, axis=1)
    sim_t = cosine_similarity(tfidf_all[idxs])
    np.fill_diagonal(sim_t, np.nan)
    tfidf_nov[idxs] = 1.0 - np.nanmean(sim_t, axis=1)
df["neural_novelty"] = neural_nov
df["tfidf_novelty"] = tfidf_nov
df.to_csv("experiment_v_results.csv", index=False)

def perm_test(a, b, n_perm=10000, seed=42):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    obs = float(np.mean(a) - np.mean(b))
    pooled = np.concatenate([a, b])
    na = len(a)
    rng_p = np.random.default_rng(seed)
    c = 0
    for _ in range(n_perm):
        rng_p.shuffle(pooled)
        if abs(float(np.mean(pooled[:na]) - np.mean(pooled[na:]))) >= abs(obs):
            c += 1
    return obs, (c + 1) / (n_perm + 1)

def holm_bonferroni(pvals):
    m = len(pvals)
    idx = np.argsort(pvals)
    adj = np.zeros(m)
    prev = 0.0
    for rank, i in enumerate(idx):
        val = min(1.0, pvals[i] * (m - rank))
        val = max(val, prev)
        adj[i] = val
        prev = val
    return adj

# Pre-declared 3-contrast primary family: v2 vs TextNudge, v2 vs RhythmMirror, v2 vs v1
lines = [
    "# Experiment V (M4R-24) — Shivering Timbers v1 (Policed) vs. v2 (Unpoliced) + Rhythm-Mirror Control (N=20/Arm)\n",
    "> **Pre-declared primary contrast family (3 contrasts per metric, Holm-Bonferroni adjusted alongside raw p):** "
    "`v2 vs TextNudge`, `v2 vs RhythmMirror`, `v2 vs v1`.\n",
    "| Condition (N=20) | Logit Entropy | Rut Escape (Neural) | Escaped Dome (%) | Neural Novelty | TF-IDF Novelty | CJK Rows |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for cond in ARMS.keys():
    g = df[df["condition"] == cond]
    lines.append(
        f"| `{cond}` | {g['mean_entropy_bits'].mean():.3f} | {g['rut_escape_neural'].mean():.4f} | "
        f"{100*g['escaped_dome'].mean():.0f}% | {g['neural_novelty'].mean():.4f} | {g['tfidf_novelty'].mean():.4f} | {int(g['has_cjk'].sum())}/20 |"
    )

lines.append("\n## Pre-Declared Contrast Family (Raw 10k Perm p + Holm-Bonferroni Adjusted p)\n")
lines.append("| Comparison | Metric | Observed Diff | Raw Perm p | Holm-Bonferroni p |")
lines.append("|---|---|---:|---:|---:|")

family_pairs = [
    ("C3_v2_Unpoliced_ZoneB_15s", "C0_TextNudgeOnly"),
    ("C3_v2_Unpoliced_ZoneB_15s", "C1_RhythmMirror_NoiseVocoded_v2"),
    ("C3_v2_Unpoliced_ZoneB_15s", "C2_v1_Policed_ZoneB_15s"),
]
for m in ["rut_escape_neural", "neural_novelty", "mean_entropy_bits"]:
    raw_res = []
    for a_n, b_n in family_pairs:
        obs, p = perm_test(df[df["condition"] == a_n][m], df[df["condition"] == b_n][m])
        raw_res.append((a_n, b_n, obs, p))
    adj_ps = holm_bonferroni([r[3] for r in raw_res])
    for (a_n, b_n, obs, p), ap in zip(raw_res, adj_ps):
        lines.append(f"| `{a_n} vs {b_n}` | `{m}` | {obs:+.4f} | {p:.5f} | {ap:.5f} |")

summary_md = "\n".join(lines) + "\n"
open("experiment_v_summary.md", "w").write(summary_md)
print(summary_md)
